# Tutorial 01 – Loading anatomy into MECHA

MECHA accepts root cross-section anatomies from two sources:

| Method | When to use |
|--------|-------------|
| **Cellset XML** | You already have an anatomy file (e.g. from GRANAR, MorphoLeaf, or a hand-drawn section exported to XML). |
| **GRANAR / granap** | You want to generate a synthetic anatomy on-the-fly using the `granap` library. |

Both methods produce the same `Mecha` object, so everything downstream (hydraulic solving, visualisation) is identical.

---

## Setup

In [ ]:
import os
import matplotlib.pyplot as plt

from openalea.mecha.mecha_class import Mecha
from openalea.mecha.utils.data_loader import InData
from openalea.mecha.utils.visu import visualize

# Ensure paths are resolved relative to this notebook's directory
TUTORIAL_DIR = os.path.dirname(os.path.abspath("tutorial_01_input.ipynb"))
INPUTS_DIR   = os.path.join(TUTORIAL_DIR, "inputs")

---
## Method 1 – Load from a cellset XML file

`InData` is the master configuration container.  
When you pass only a `cellset_file`, all other parameters (hydraulics, geometry, boundary conditions) use sensible defaults that you can override later.

In [ ]:
cellset_path = os.path.join(INPUTS_DIR, "current_root.xml")

AllIn = InData(cellset_file=cellset_path)
AllIn.info()

### Build the network and visualise the cross-section

Instantiating `Mecha` parses the cellset, builds the hydraulic network graph, and assigns default hydraulic properties to each wall/membrane.

In [ ]:
mecha_xml = Mecha(AllIn)

visualize(mecha_xml, visu_type="polygon")

### Compute whole-root hydraulic conductivities

`compute_conductivities()` solves two pressure systems (one for radial flow, one for axial flow) and returns the macroscopic radial conductivity **kr** [cm hPa⁻¹ s⁻¹] and axial conductance **Kx** [cm³ hPa⁻¹ s⁻¹] for each maturity stage defined in `AllIn.geometry`.

In [ ]:
mecha_xml.compute_conductivities()

print("Hydraulic properties (cellset XML input):")
for stage in mecha_xml.root_hydraulic_properties:
    print(f"  Barrier type {stage['barrier']:>2d} | "
          f"kr = {stage['kr']:.4e} cm hPa⁻¹ s⁻¹ | "
          f"Kx = {stage['Kx']:.4e} cm³ hPa⁻¹ s⁻¹")

---
## Method 2 – Load via GRANAR (`granap`)

GRANAR generates synthetic root cross-sections from a small set of anatomical parameters.  
The `granap` library (the Python reimplementation of GRANAR) is accessed through `RootAnatomy`.

> **Prerequisite:** `granap` requires [`pydantic`](https://docs.pydantic.dev/).  
> Install it with `pip install pydantic` if the import below fails.

The workflow is:
1. Create a `RootAnatomy` (uses sensible defaults).
2. Optionally tweak parameters with `update_params`.
3. Call `export_to_adjencymatrix()` to build the internal cell adjacency graph.
4. Wrap the anatomy in `NetworkBuilder` and pass it to `Mecha`.

In [ ]:
from openalea.granap.root_class import RootAnatomy
from openalea.mecha.utils.network_builder import NetworkBuilder

### Generate the anatomy

Here we create a simple maize-like root (the default) with a small proportion of aerenchyma.

In [ ]:
root = RootAnatomy()

# Optional: customise anatomy parameters before generating cells
root.update_params("aerenchyma", "aerenchyma_proportion", 0.02)
root.update_params("aerenchyma", "n_files", 1)
root.update_params("inter_cellular_spaces", "tissue", "cortex")

# Build the cell adjacency matrix (required before passing to MECHA)
_ = root.export_to_adjencymatrix()

print(f"Generated anatomy: {len(root.all_cells.cells)} cells")

### Build the MECHA network from the GRANAR anatomy

In [ ]:
# Default InData (all hydraulic & geometry parameters use built-in defaults)
default_input = InData()

# Set two maturity stages: Casparian strip only (barrier=1) then full suberin (barrier=3)
default_input.geometry.set_maturity_stages([1, 3])

# Wrap the GRANAR anatomy in a NetworkBuilder and populate it
granar_network = NetworkBuilder(root)
granar_network.populate_from_network()

# Pass the pre-built network to Mecha
mecha_granar = Mecha(default_input, network=granar_network)

### Visualise the GRANAR cross-section

In [ ]:
visualize(mecha_granar, visu_type="polygon")

### Compute hydraulic conductivities

In [ ]:
mecha_granar.compute_conductivities()

print("Hydraulic properties (GRANAR input):")
for stage in mecha_granar.root_hydraulic_properties:
    print(f"  Barrier type {stage['barrier']:>2d} | "
          f"kr = {stage['kr']:.4e} cm hPa⁻¹ s⁻¹ | "
          f"Kx = {stage['Kx']:.4e} cm³ hPa⁻¹ s⁻¹")

---
## Side-by-side comparison

Both methods feed into the same MECHA solver, so you can compare the polygonal anatomy and the resulting kr / Kx values side by side.

In [ ]:
from openalea.mecha.utils.visu import prep_section, plot_organ_section

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, mecha_obj, title in [
    (axes[0], mecha_xml,   "Method 1 – Cellset XML"),
    (axes[1], mecha_granar, "Method 2 – GRANAR (granap)"),
]:
    gdf = mecha_obj.network._cells_gdf
    gdf.plot(
        ax=ax,
        column="type",
        cmap="viridis",
        edgecolor="black",
        linewidth=0.5,
        alpha=0.6,
        legend=True,
        legend_kwds={"title": "Cell type", "loc": "lower right", "fontsize": 8},
    )
    ax.set_aspect("equal", "box")
    ax.set_title(title, fontsize=12)
    ax.set_xlabel("x (µm)")
    ax.set_ylabel("y (µm)")

plt.tight_layout()
plt.show()

---
## Exploring Advanced Visualization Methods

MECHA provides a rich set of visualization tools in `mecha.utils.visu` to inspect the hydraulic networks, potentials, conductances, and flows.

Below, we demonstrate the available visualization types using `mecha_granar` (maturity stage index `1`, corresponding to fully suberized endodermis):

### 1. Water potential map

Plots a heatmap showing the computed water potentials (in hPa) in each cell of the root cross-section.

In [ ]:
visualize(mecha_granar, visu_type="water_potential", maturity_idx=1)

### 2. Conductance Network

Generates a three-panel plot displaying the log-scaled hydraulic conductance ($K$) of edges across the apoplastic (cell wall), transcellular (membrane), and symplastic (plasmodesmata) pathways.

In [ ]:
visualize(mecha_granar, visu_type="conductance", maturity_idx=1)

### 3. Radial Potential Profile

Plots the average water potential vs. the radial position (from the root center to the epidermis) to inspect the pressure drop across different tissue layers.

In [ ]:
visualize(mecha_granar, visu_type="psi_profile", maturity_idx=1)

### 4. Radial Flow Pathways Breakdown

Shows the relative contribution (percentage) of the apoplastic, transcellular, and symplastic pathways to the total radial water flow across different cell layers.

In [ ]:
visualize(mecha_granar, visu_type="flow_pathway", maturity_idx=1)

### 5. Water Flow Vectors

Draws arrows on the network graph edges, with sizes proportional to the water flow magnitude $|Q|$ and directions matching the actual flow direction.

In [ ]:
visualize(mecha_granar, visu_type="flow", maturity_idx=1)

### 6. Water Velocity Vectors

Similar to flow vectors, but the arrows are scaled by the flow velocity.

In [ ]:
visualize(mecha_granar, visu_type="velocity", maturity_idx=1)

### 8. ParaView Export

Exports the solved network and its physical geometries as VTK files, allowing high-fidelity 3D visualization and post-processing in ParaView.

In [ ]:
# Creates VTK files under the outputs directory
os.makedirs("outputs", exist_ok=True)
visualize(mecha_granar, visu_type="paraview", maturity_idx=1, prefix="outputs/sim")